# Predict Edges

In [ ]:

from collections import Counter
import jsonlines
import pandas as pd
import numpy as np
from numpy.typing import ArrayLike
import os

from ignite.engine import Engine, Events
from ignite.handlers import ProgressBar
from ignite.metrics import Recall, Precision, Accuracy, Loss

import torch
from torch import nn
import torch.nn.functional as F
import torch_geometric
from torch_geometric.data import HeteroData
from torch_geometric.nn import GraphConv, SAGEConv, to_hetero, HeteroConv
from torch_geometric import transforms as T
from torch_geometric import seed_everything

from jazz_graph.data.reporting import inspect_degrees
from jazz_graph.training.logging import (
    ExperimentLogger,
    save_embeddings_handler,
    save_checkpoint_handler,
    run_evaluator_handler,
    log_experiment_handler,
    console_logging,
    binary_output_transform
)
from jazz_graph.data.graph_builder import CreateTensors, prune_island_nodes, make_jazz_data
from jazz_graph.model.model import JazzModel, LinkPredictionModel, NodeClassifier


In [ ]:
models_dir = '/workspace/local_data/graph_parquet_proto'
assert os.path.exists(models_dir)
create = CreateTensors(models_dir)

In [ ]:
data = make_jazz_data(create)

In [ ]:
# TODO: report on the data a little more concreately.
# E.g., who are the hub nodes? How many nodes have > 50 edges.
# how many nodes have < 6 edges? All these, by type.
# Get really fancy and visualize a sub-graph.

def frequency_of_n_labels(data: HeteroData):
    """Return frequency of number of labels in the data, i.e., what percentage have 1 label, 0 labels, etc."""
    count_by_row = data['performance'].y.sum(dim=1)
    n_samples = data['performance'].y.shape[0]
    counter = Counter((int(x) for x in (count_by_row)))
    for i in range(len(counter)):
        count = counter[i]
        freq = count / n_samples
        print(f"Num samples with {i} labels: {freq:.3f}")

In [ ]:
data.metadata()

In [ ]:
create.label_names()

In [ ]:
print(data)
print(
    f"The graph contains {'' if data.has_isolated_nodes() else 'no '}isolated nodes and",
    f"is {'directed' if data.is_directed() else 'undirected'}."
)
frequency_of_n_labels(data)
for style, count in (zip(create.label_names(), data['performance'].y.sum(dim=0))):
    print(f"  {style}: {int(count) / create._labels.shape[0]:.1%}")
    # Easy Listening is probably a mislabel by modern standards.


In [ ]:
inspect_degrees(data)

## Model

## Edge Prediction

Quick shot at writing an edge prediction model. 
Conceptually, a recommender system based on this prediction "who worked with whom, on what?"
Thus, it's a lot like asking Theolious Monk "What would you recommend?"

In [ ]:
seed_everything(42)
performs_edge_count = data[('artist', 'performs', 'performance')].num_edges

data_config = {
    'dataset': models_dir,
    'val_frac': .6,
    'test_frac': .1,
    'disjoint_train_ratio': 0.3,
    'neg_sample_ratio': 6.0,
    'sampling': {'num_neighbors': [5]}
}

split_graph = T.RandomLinkSplit(
    num_val=int(performs_edge_count * data_config['val_frac']),
    num_test=int(performs_edge_count * data_config['test_frac']),
    disjoint_train_ratio=data_config['disjoint_train_ratio'],
    # neg_sampling_ratio=data_config['neg_sample_ratio'],
    add_negative_train_samples=False,
    edge_types=[('artist', 'performs', 'performance')],
    rev_edge_types=[('performance', 'rev_performs', 'artist')],
    is_undirected=True
)
train_data, dev_data, test_data = split_graph(data)

In [ ]:
for edge in data.metadata()[1]:
    edges = data[edge].edge_index
    print(edge)
    unique = len(set(map(tuple, edges.t().tolist())))
    print("unique:", unique)
    print("total:", edges.shape[1])
    print("ratio:", (edges.shape[1] - unique) / edges.shape[1])

In [ ]:

def shared_label_edges(edge_data: HeteroData, edge_label_data: HeteroData, edge: str | tuple):
    d1_edges = {tuple(e) for e in edge_data[edge].edge_label_index.t().tolist()}
    d2_edges = {tuple(e) for e in edge_label_data[edge].edge_label_index.t().tolist()}
    shared_edges = d1_edges.intersection(d2_edges)
    return shared_edges

def shared_rev_edges(data1, data2):
    train_edges = {
        (a,b) for a,b in data1['performs'].edge_index.t().tolist()
    }

    val_edges = {
        (a,b) for a,b in data2['performs'].edge_label_index.t().tolist()
    }

    reverse_overlap = {(b,a) for (a,b) in val_edges} & train_edges
    print(len(reverse_overlap))

shared_rev_edges(train_data, dev_data)
shared = shared_label_edges(train_data, dev_data, 'performs')
assert not shared, f"There are {len(shared)} edges in train-dev performs."


In [ ]:
from torch_geometric.loader import LinkNeighborLoader

def edge_training_data_factory(data: HeteroData, data_config: dict) -> LinkNeighborLoader:
    sampling = data_config['sampling']
    edge_loader = LinkNeighborLoader(
        data=data,
        num_neighbors=sampling['num_neighbors'],
        neg_sampling_ratio=data_config['neg_sample_ratio'],
        edge_label_index=(('artist', 'performs', 'performance'), data['performs'].edge_label_index),
        edge_label=data['performs'].edge_label,
        batch_size=128,
        shuffle=True
    )
    return edge_loader

edge_loader_train = edge_training_data_factory(train_data, data_config)
edge_loader_dev = edge_training_data_factory(dev_data, data_config)

In [ ]:
iter_edges = iter(edge_loader_dev)

In [ ]:
batch = next(iter_edges)
label_edges = {
    tuple(e) for e in batch['artist','performs','performance']
    .edge_label_index.t().tolist()
}
mp_edges = {
    tuple(e) for e in batch['artist','performs','performance']
    .edge_index.t().tolist()
}
leaked = label_edges & mp_edges
print("Leaked edges:", len(leaked))

reverse_mp = {(b,a) for (a,b) in mp_edges}
reverse_leak = label_edges & reverse_mp
print("Reverse leaks:", len(reverse_leak))

In [ ]:
model_config = {
    'num_performances': data['performance'].num_nodes,
    'num_artists': data['artist'].num_nodes,
    'num_songs': data['song'].num_nodes,
    'hidden_dim': 128,
    'embed_dim': 64,
    'dropout': 0.
}

model = LinkPredictionModel(JazzModel(
    model_config['num_performances'],
    model_config['num_artists'],
    model_config['num_songs'],
    hidden_dim=model_config['hidden_dim'],
    embed_dim=model_config['embed_dim'],
    dropout=model_config['dropout'],
    metadata=data.metadata()
))

batch = next(iter(edge_loader_train))
batch['performance'].n_id
batch['performs'].edge_label
batch['performs'].edge_label_index.shape
model(batch).shape


In [ ]:
class GNNTrainingLogic:
    """Define training step and eval steps."""
    def __init__(self, model, optimizer, criterion):
        self.device = next(model.parameters()).device
        self.model = model
        self.optimizer = optimizer
        self.criterion = criterion

    def _extract_model_args(self, batch):
        return batch.x_dict, batch.edge_index_dict, batch['performs'].edge_label_index

    def train_step(self, engine, batch: HeteroData) -> dict:
        """Complete one step of gradient descent."""
        self.model.train()
        self.optimizer.zero_grad()
        batch.to(self.device)

        y_pred = self.model(batch)
        y_true = batch['performs'].edge_label
        loss = self.criterion(y_pred, y_true)
        loss.backward()
        self.optimizer.step()
        return {'loss': loss.item(), 'y_pred': y_pred.detach(), 'y_true': y_true.detach()}

    def eval_step(self, engine, batch: HeteroData) -> dict:
        """Complete one pass over a batch of data with no-grad and return results."""
        self.model.eval()
        batch.to(self.device)
        with torch.no_grad():
            y_pred = self.model(batch)
            y_true = batch['performs'].edge_label
        return {'y_pred': y_pred, 'y_true': y_true}

experiment_config = {
    'data_config': data_config,
    'model': model_config,
    'lr': .001,
    'batch_size': edge_loader_train.batch_size,
    'epochs': 3
}


criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=experiment_config['lr'])
trainer_logic = GNNTrainingLogic(model, optimizer, criterion)
trainer_logic.train_step(None, batch)
None # don't push to stdout


In [ ]:
from jazz_graph.training.logging import save_checkpoint_handler


experiment_logger = ExperimentLogger(root='../experiments', run_name=f'gnn_link_prediction_{os.path.basename(models_dir)}', config=experiment_config)

trainer = Engine(trainer_logic.train_step)
dev_evaluator = Engine(trainer_logic.eval_step)

metrics = {
    'accuracy': Accuracy(output_transform=binary_output_transform),
    'recall': Recall(output_transform=binary_output_transform),
    'precision': Precision(output_transform=binary_output_transform),
    'loss': Loss(criterion, output_transform=lambda out: (out['y_pred'], out['y_true']))
}

for name, metric in metrics.items():
    metric.attach(trainer, name)
    metric.attach(dev_evaluator, name)

progress_bar = ProgressBar()
progress_bar.attach(trainer)

trainer.add_event_handler(Events.EPOCH_COMPLETED, console_logging, 'Training', trainer)
trainer.add_event_handler(Events.EPOCH_COMPLETED, run_evaluator_handler, dev_evaluator, edge_loader_train)
trainer.add_event_handler(Events.EPOCH_COMPLETED, log_experiment_handler, experiment_logger, 'train', trainer)
dev_evaluator.add_event_handler(Events.EPOCH_COMPLETED, log_experiment_handler, experiment_logger, 'dev', trainer)
dev_evaluator.add_event_handler(Events.EPOCH_COMPLETED, console_logging, 'Valiation', trainer)
trainer.add_event_handler(Events.COMPLETED, save_checkpoint_handler, experiment_logger, model)
trainer.add_event_handler(Events.COMPLETED, save_embeddings_handler, experiment_logger, model)


In [ ]:
trainer.run(edge_loader_train, max_epochs=15)